In [45]:
import requests, pandas as pd
#quote is the url encoder
from urllib.parse import quote
from eea import disco

In [46]:
#TBL so I don't have to rewrite every time
TBL = "[CO2Emission].[latest].[co2cars_2025Pv31]"

So now we'll do some EDA to figure out which columns are necessary. Which specs will I want to predict WLTP CO2 from?
First I'll just get a quick overview of the data by looking at some inital rows.

In [47]:
disco(f"SELECT TOP 100 * "
f"FROM {TBL}")

,ID,MS,Mp,VFN,Mh,Man,MMS,TAN,T,Va,...,Year,Status,Version_file,E (g/km),Er (g/km),Zr,Dr,Fc,Ech,RLFI
0,175878466,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-12-22,5.2,EA,RL-050009-NLH
1,175901181,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-05-26,5.2,EA,RL-050009-NLH
2,176038548,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-12-19,5.2,EA,RL-050009-NLH
3,176039670,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-05-28,5.2,EA,RL-050009-NLH
4,176052774,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-08-25,5.2,EA,RL-050009-NLH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,176512774,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-10-01,5.2,EA,RL-050009-NLH
96,176514993,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-12-09,5.2,EA,RL-050009-NLH
97,176540477,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-02-03,5.2,EA,RL-050009-NLH
98,176545927,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-06-20,5.2,EA,RL-050009-NLH


In [ ]:
#How many rows are there?
disco(f"SELECT COUNT(*) AS n_rows " 
f"FROM {TBL}")

,n_rows
0,10833597


In [49]:
#target distribution (name the aggregate, or it errors)
Ewltp = "[Ewltp (g/km)]"
disco(f"SELECT Ft AS fuel_type, COUNT(*) AS n, AVG(CAST({Ewltp} AS float)) AS mean_co2 " 
f"FROM {TBL}" 
f"GROUP BY Ft " 
f"ORDER BY n DESC")

,fuel_type,n,mean_co2
0,petrol,6075402,128.027738
1,electric,2055863,0.000000
2,diesel,1279640,151.832064
3,petrol/electric,1009882,31.796999
4,lpg,352268,119.812034
5,diesel/electric,47590,36.344846
6,e85,12384,128.497253
7,hydrogen,451,0.000000
8,ng,108,107.240741
9,unknown,9,184.285714


In [53]:
#cardinality of the scary columns
disco(f"SELECT COUNT(DISTINCT Mk) AS n_Mk_makes, COUNT(DISTINCT Cn) AS n_Cn_names " 
f"FROM {TBL}")

,n_Mk_makes,n_Cn_names
0,500,7273


Now we'll get into inspecting some impotant columns more closely to see if there's anything wrong with them, or to see if they need to be adressed before we can train the model.
This includes missing data or potential data leakage that can impact the accuracy of our model.

In [51]:
#missingness, one column at a time
disco(f"SELECT COUNT(*) AS n_null " 
f"FROM {TBL}" 
f"WHERE {Ewltp} IS NULL")

,n_null
0,6526


In [55]:
disco(f"SELECT COUNT(*) AS not_null " 
f"FROM {TBL}"
f"WHERE {Ewltp} IS NOT NULL")

,not_null
0,10827071


So Ewltp (g/km) has about 0.06% missing values.

In [52]:
Z = "[Z (Wh/km)]"
disco(f"SELECT {Z}"
f"FROM {TBL}")

,Z (Wh/km)
0,None
1,None
2,None
3,None
4,None
...,...
995,None
996,None
997,None
998,None


In [56]:
disco(f"SELECT COUNT(*) AS n_null " 
f"FROM {TBL}" 
f"WHERE {Z} IS NULL")

,n_null
0,7885389


So electric energy consumption (Z (Wh/km)) doesn't appear to have any useful information. The vast majority of its rows are empty, so we won't be using it to train our model.

Used DuckDB bc embedded OLAP (column store, analytical)